# Darcy across a permeability jump

A vertical material interface separates permeabilities 1 and 1000. The exact pressure has a gradient jump, while the normal physical flux is continuous. The interface is aligned with macro and fine meshes; this is not an unfitted benchmark.

In [ ]:
from pathlib import Path
import sys

# Locate the checkout from the repository root or any notebook subdirectory.
ROOT = next(
    (path for path in (Path.cwd(), *Path.cwd().parents)
     if (path / "pyproject.toml").is_file()
     and (path / "src" / "pymhm").is_dir()
     and (path / "notebooks").is_dir()),
    None,
)
if ROOT is None:
    raise RuntimeError("Run this notebook from the PyMHM checkout.")
root = ROOT
for directory in (ROOT, ROOT / "examples"):
    if str(directory) not in sys.path:
        sys.path.insert(0, str(directory))


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from pymhm import TriangleMesh, SkeletonSpace, FaceSpace
from examples.formulations.application import brinkman as solve_brinkman
from examples.formulations.application import elasticity as solve_elasticity


## Declare the local and global equations

The application writes local A, B and C blocks with the shared FEM kernels,
then supplies them to the generic assembler. Primal coefficients satisfy

$$
\begin{aligned}
a_T(p,v)&=(K\nabla p,\nabla v)_T,\\
L_T(v)&=(f,v)_T+\sum_j\alpha_jv(x_j),\\
B_T\lambda&=\langle\lambda,v\rangle_{\partial T},\\
C_Tp&=-\langle p,\mu\rangle_{\partial T}.
\end{aligned}
$$

Signs follow the fixed macroface normals. RT0 instead uses flux, pressure and
private boundary-pressure coefficients, including the local normal-flux
constraint. The global `Equation` supplies the signed Dirichlet moments and
retains each constant pressure mode. Pure Neumann data add the physical pressure
integral through an explicit constraint. `define_darcy` returns equations;
`assemble`, the global solve and field interpretation are separate stages.


In [ ]:
from pymhm import assemble, SolverConfig
from examples.formulations.darcy import define_darcy, pressure_constraints, recover_darcy

from pathlib import Path
import sys

sys.path.insert(0, str(root / "examples"))
from plot_mesh import draw_macro_mesh

mesh = TriangleMesh.unit_square(2)


def permeability(x):
    """Return the scalar permeability in each material."""
    return np.where(x[:, 0] < 0.5, 1.0, 1000.0)


def pressure(x):
    """Return a continuous piecewise affine pressure with unit horizontal flux."""
    return np.where(x[:, 0] <= 0.5, 1 - x[:, 0], 0.5 - (x[:, 0] - 0.5) / 1000)


result_definition = define_darcy(mesh, permeability=permeability, dirichlet=pressure)
result_system = assemble(result_definition.problem)
result = recover_darcy(
    result_definition,
    result_system,
    result_system.solve(constraints=pressure_constraints(result_definition, result_system)),
)
assert result.l2_error(pressure) < 1e-11
assert result.flux_l2_error([1, 0]) < 1e-09
print("pressure error", result.l2_error(pressure), "flux error", result.flux_l2_error([1, 0]))
figure, axis = plt.subplots(figsize=(6, 5), constrained_layout=True)
for local, p in zip(result.local_meshes, result.pressure):
    artist = axis.tripcolor(
        local.points[:, 0],
        local.points[:, 1],
        local.cells,
        p,
        shading="gouraud",
        vmin=0.4995,
        vmax=1,
    )
figure.colorbar(artist, ax=axis, label="Pressure")
draw_macro_mesh(axis, mesh, label=True)
axis.legend(loc="lower center", bbox_to_anchor=(0.5, 1.03))
axis.set(xlabel="x", ylabel="y", aspect="equal")
plt.show()


A material tensor must be symmetric positive definite at quadrature points. Resolving the true coefficient interfaces and oscillation length remains a discretization requirement.